# QA 08 — complete neural figure gallery

Every public neural Plotly figure, synthetic and real datasets, simple and complex topology, extensibility, scale, fallbacks, and recorder schema v2.

> Run this notebook from top to bottom after installing the project with
> `pip install -e ".[notebooks]"`. Figures are genuine public-API outputs.
> Cells intentionally contain no assertions: automated invariants live in
> `tests/`, while this notebook is for human visual inspection.

In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
if not (candidate / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from inside the Mlektic repository.")
ROOT = candidate
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


from IPython.display import display
import numpy as np
import torch
from sklearn.datasets import load_breast_cancer, load_digits, load_iris
from sklearn.preprocessing import StandardScaler
from mlektic import (TorchTrainingRecorder, explain_nn_prediction,
                     register_neural_descriptor, visualize_nn,
                     visualize_nn_architecture, visualize_nn_blocks,
                     visualize_nn_backpropagation, visualize_nn_graph,
                     visualize_nn_hyperparameters,
                     visualize_nn_loss_landscape, visualize_nn_training,
                     visualize_nn_weights)
from notebooks._support import case_heading, torch_xor_case

torch.manual_seed(17)

class ResidualNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.first = torch.nn.Linear(4, 4)
        self.activation = torch.nn.ReLU()
        self.second = torch.nn.Linear(4, 4)
    def forward(self, x):
        return x + self.second(self.activation(self.first(x)))

class SharedNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = torch.nn.Linear(4, 4)
    def forward(self, x):
        return self.shared(x) + self.shared(x)

class ConvNet(torch.nn.Module):
    def __init__(self, classes=3, in_channels=3):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels, 4, 3, padding=1),
            torch.nn.BatchNorm2d(4),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
        )
        self.classifier = torch.nn.Linear(4 * 4 * 4, classes)
    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

class EmbeddingNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(20, 6)
        self.output = torch.nn.Linear(6, 3)
    def forward(self, token_ids):
        return self.output(self.embedding(token_ids).mean(dim=1))

class SiameseNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = torch.nn.Linear(4, 3)
        self.head = torch.nn.Linear(6, 2)
    def forward(self, left, right):
        left_state = torch.relu(self.encoder(left))
        right_state = torch.relu(self.encoder(right))
        score = self.head(torch.cat((left_state, right_state), dim=-1))
        return score, left_state, right_state

class DynamicBranch(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.positive = torch.nn.Linear(4, 2)
        self.negative = torch.nn.Linear(4, 2)
    def forward(self, x):
        return self.positive(x) if x.sum().item() > 0 else self.negative(x)

def record_case(model, X, y, loss_fn, *, task, steps=10, learning_rate=0.03):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    recorder = TorchTrainingRecorder(
        model,
        optimizer=optimizer,
        loss_fn=loss_fn,
        capture_optimizer_state=True,
    )
    for step in range(steps):
        optimizer.zero_grad()
        prediction = model(X)
        loss = loss_fn(prediction, y)
        loss.backward()
        optimizer.step()
        recorder.record(
            step + 1,
            loss=loss,
            predictions=prediction,
            targets=y,
            task=task,
            capture_phase="post_step",
        )
    recorder.close()
    return model, X, recorder.to_history()

# Small synthetic binary classification.
xor_model, xor_X, xor_history = torch_xor_case(steps=10)

# Synthetic nonlinear regression.
generator = torch.Generator().manual_seed(17)
reg_X = torch.rand((96, 2), generator=generator) * 4.0 - 2.0
reg_y = (0.7 * reg_X[:, :1] ** 2 - 0.4 * reg_X[:, 1:] + torch.sin(reg_X[:, :1]))
reg_model, reg_X, reg_history = record_case(
    torch.nn.Sequential(torch.nn.Linear(2, 10), torch.nn.Tanh(), torch.nn.Linear(10, 1)),
    reg_X,
    reg_y,
    torch.nn.MSELoss(),
    task="regression",
    steps=12,
)

# Real Iris multiclass classification (bundled with Scikit-learn; no download).
iris = load_iris()
iris_X = torch.tensor(StandardScaler().fit_transform(iris.data), dtype=torch.float32)
iris_y = torch.tensor(iris.target, dtype=torch.long)
iris_model, iris_X, iris_history = record_case(
    torch.nn.Sequential(torch.nn.Linear(4, 12), torch.nn.ReLU(), torch.nn.Linear(12, 3)),
    iris_X,
    iris_y,
    torch.nn.CrossEntropyLoss(),
    task="classification",
    steps=12,
)

# Real breast-cancer binary classification.
cancer = load_breast_cancer()
cancer_X = torch.tensor(StandardScaler().fit_transform(cancer.data), dtype=torch.float32)
cancer_y = torch.tensor(cancer.target[:, None], dtype=torch.float32)
cancer_model, cancer_X, cancer_history = record_case(
    torch.nn.Sequential(
        torch.nn.Linear(cancer_X.shape[1], 16),
        torch.nn.ReLU(),
        torch.nn.Dropout(0.1),
        torch.nn.Linear(16, 1),
        torch.nn.Sigmoid(),
    ),
    cancer_X,
    cancer_y,
    torch.nn.BCELoss(),
    task="classification",
    steps=10,
)

# Real handwritten digits represented as 8x8 grayscale images.
digits = load_digits()
digits_X = torch.tensor(digits.images[:240, None] / 16.0, dtype=torch.float32)
digits_y = torch.tensor(digits.target[:240], dtype=torch.long)
digits_model, digits_X, digits_history = record_case(
    ConvNet(classes=10, in_channels=1),
    digits_X,
    digits_y,
    torch.nn.CrossEntropyLoss(),
    task="classification",
    steps=8,
    learning_rate=0.02,
)

This is the exhaustive neural figure gallery. It invokes every public Plotly
figure route with progressively richer models and both synthetic and real
datasets. The real datasets are bundled with Scikit-learn and require no
network download. Hover blocks to inspect shapes, parameters, buffers,
hyperparameters, readable mathematics, and capture provenance.

Training fixtures use short deterministic full-batch runs to create genuine
recorded states for visual inspection. They are not train/test benchmarks and
must not be interpreted as generalization estimates.

`build_nn_math_report`, `display_nn_math_report`, and `export_nn_math_report`
produce HTML reports rather than Plotly figures, so they remain covered by the
report tests and the dedicated neural documentation instead of this figure
gallery.

## Every `visualize_nn` figure route — synthetic XOR

### `NN-ROUTER-ARCHITECTURE`

**Inspect:** the generic router's legacy architecture view

In [ ]:
case_heading("NN-ROUTER-ARCHITECTURE", "the generic router's legacy architecture view")
display(visualize_nn(xor_model,xor_X[:1],history=xor_history,view='architecture',theme='academic'))

### `NN-ROUTER-BLOCKS`

**Inspect:** the generic router's execution-block view

In [ ]:
case_heading("NN-ROUTER-BLOCKS", "the generic router's execution-block view")
display(visualize_nn(xor_model,xor_X[:1],view='blocks',theme='classroom',size='wide'))

### `NN-ROUTER-HYPERPARAMETERS`

**Inspect:** every effective model, optimizer-group, objective, and scheduler hyperparameter with its PyTorch-aligned mathematical definition

In [ ]:
case_heading("NN-ROUTER-HYPERPARAMETERS", "every effective model, optimizer-group, objective, and scheduler hyperparameter with its PyTorch-aligned mathematical definition")
hyper_model=torch.nn.Sequential(torch.nn.Linear(4,8,bias=False),torch.nn.BatchNorm1d(8,eps=1e-4,momentum=0.2),torch.nn.LeakyReLU(0.15),torch.nn.Dropout(0.25),torch.nn.Linear(8,3))
hyper_optimizer=torch.optim.Adam(hyper_model.parameters(),lr=0.002,betas=(0.8,0.95),weight_decay=0.01)
hyper_objective=torch.nn.CrossEntropyLoss(label_smoothing=0.05)
hyper_scheduler=torch.optim.lr_scheduler.StepLR(hyper_optimizer,step_size=3,gamma=0.4)
display(visualize_nn_hyperparameters(hyper_model,optimizer=hyper_optimizer,loss_fn=hyper_objective,scheduler=hyper_scheduler,theme='academic',size='wide'))

### `NN-ROUTER-GRAPH`

**Inspect:** the generic router's fluid hybrid animation with loss but without the optional backpropagation overlay

In [ ]:
case_heading("NN-ROUTER-GRAPH", "the generic router's fluid hybrid animation with loss but without the optional backpropagation overlay")
display(visualize_nn(xor_model,xor_X[1],history=xor_history,view='graph',max_frames=None,frame_duration=360,evolution_mode='hybrid',update_reference='previous',update_scale='global',show_update_panel=False,show_loss_panel=True,show_backpropagation=False,top_k_updates=6,interpolation_frames=3,math_font_scale=1.15))

### `NN-ROUTER-TRAINING`

**Inspect:** the generic router's recorded loss and metric panels

In [ ]:
case_heading("NN-ROUTER-TRAINING", "the generic router's recorded loss and metric panels")
display(visualize_nn(xor_model,history=xor_history,view='training',max_frames=6,theme='academic'))

### `NN-ROUTER-WEIGHTS`

**Inspect:** the generic router's evolving parameter matrices

In [ ]:
case_heading("NN-ROUTER-WEIGHTS", "the generic router's evolving parameter matrices")
display(visualize_nn(xor_model,history=xor_history,view='weights',max_frames=6,theme='academic'))

### `NN-ROUTER-ACTIVATIONS`

**Inspect:** the generic router's recorded activation vectors

In [ ]:
case_heading("NN-ROUTER-ACTIVATIONS", "the generic router's recorded activation vectors")
display(visualize_nn(xor_model,xor_X[:1],history=xor_history,view='activations',max_frames=6,theme='accessible'))

### `NN-ROUTER-BACKPROPAGATION`

**Inspect:** the generic router's chain-rule view with recorded layer-gradient norms

In [ ]:
case_heading("NN-ROUTER-BACKPROPAGATION", "the generic router's chain-rule view with recorded layer-gradient norms")
display(visualize_nn(xor_model,xor_X[:1],history=xor_history,view='backpropagation',max_frames=6,theme='academic',math_font_scale=1.2))

### `NN-TRAINING-QUERY-REPLAY`

**Inspect:** an independent replay of recorded parameter and signal evolution without prediction cards

In [ ]:
case_heading("NN-TRAINING-QUERY-REPLAY", "an independent replay of recorded parameter and signal evolution without prediction cards")
display(explain_nn_prediction(xor_model,xor_X[1],history=xor_history,max_frames=6,parameter_state='training_replay',theme='academic'))

### `NN-GALLERY-FORWARD-SUBSTITUTION`

**Inspect:** a final-model input, numerical substitution, and output lesson for one XOR observation

In [ ]:
case_heading("NN-GALLERY-FORWARD-SUBSTITUTION", "a final-model input, numerical substitution, and output lesson for one XOR observation")
display(explain_nn_prediction(xor_model,xor_X[1],history=xor_history,parameter_state='final',theme='academic',size='wide'))

### `NN-GALLERY-GRAPH-SIGNAL`

**Inspect:** smooth relative-activation contrast with forward-signal edge coloring

In [ ]:
case_heading("NN-GALLERY-GRAPH-SIGNAL", "smooth relative-activation contrast with forward-signal edge coloring")
display(visualize_nn_graph(xor_model,xor_X[1],xor_history,max_frames=None,frame_duration=360,interpolation_frames=3,node_color_mode='relative',edge_color_mode='signal',theme='accessible'))

### `NN-GALLERY-REPORT-FIGURE`

**Inspect:** a static reduced-motion prediction figure for publication

In [ ]:
case_heading("NN-GALLERY-REPORT-FIGURE", "a static reduced-motion prediction figure for publication")
display(explain_nn_prediction(xor_model,xor_X[2],history=xor_history,format='report',reduced_motion=True,size='wide'))

## Dense graph performance: without and with recorded backpropagation

The default graph omits the per-edge gradient overlay. This reduces animated
trace count while preserving forward activity, parameters, parameter updates,
and optional loss. The second case explicitly enables the dotted reverse-mode
gradient traces. Compare playback in the same notebook environment.

### `NN-GRAPH-WITHOUT-BACKPROP`

**Inspect:** the default fluid graph with evolving parameters and loss but without per-edge gradient traces

In [ ]:
case_heading("NN-GRAPH-WITHOUT-BACKPROP", "the default fluid graph with evolving parameters and loss but without per-edge gradient traces")
display(visualize_nn_graph(xor_model,xor_X[1],xor_history,max_frames=None,frame_duration=360,interpolation_frames=3,show_backpropagation=False,show_loss_panel=True,theme='academic',size='wide'))

### `NN-GRAPH-WITH-BACKPROP`

**Inspect:** the same graph with optional recorded reverse-mode gradients

In [ ]:
case_heading("NN-GRAPH-WITH-BACKPROP", "the same graph with optional recorded reverse-mode gradients")
display(visualize_nn_graph(xor_model,xor_X[1],xor_history,max_frames=None,frame_duration=520,interpolation_frames=3,show_backpropagation=True,theme='academic',size='wide'))

## Parameter-update evolution modes

The classic `absolute` graph remains the default. The following opt-in views
make small parameter changes perceptually visible without replacing their
mathematical values. A signed halo encodes the actual difference between the
current parameters and the selected reference; its width and opacity encode
the magnitude. Dashed edges continue to represent recorded gradients.

`update_scale="global"` keeps magnitudes comparable throughout the animation.
`update_scale="frame"` deliberately renormalizes every frame for contrast, so
its color intensity must not be compared across time. Interpolated frames make
motion smoother but are labeled as perceptual states, not optimizer steps; the
slider contains only recorded checkpoints.

### `NN-GRAPH-HYBRID-UPDATES`

**Inspect:** absolute weights plus globally comparable signed update halos and smooth perceptual motion

In [ ]:
case_heading("NN-GRAPH-HYBRID-UPDATES", "absolute weights plus globally comparable signed update halos and smooth perceptual motion")
display(visualize_nn_graph(xor_model,xor_X[1],xor_history,max_frames=None,frame_duration=360,evolution_mode='hybrid',update_reference='previous',update_scale='global',top_k_updates=6,interpolation_frames=3,theme='academic',size='wide'))

### `NN-GRAPH-CUMULATIVE-UPDATES`

**Inspect:** parameter displacement from the initial checkpoint without the absolute-weight color encoding

In [ ]:
case_heading("NN-GRAPH-CUMULATIVE-UPDATES", "parameter displacement from the initial checkpoint without the absolute-weight color encoding")
display(visualize_nn_graph(xor_model,xor_X[2],xor_history,max_frames=6,frame_duration=320,evolution_mode='updates',update_reference='initial',update_scale='global',interpolation_frames=2,theme='classroom',size='wide'))

### `NN-GRAPH-FRAME-NORMALIZED-UPDATES`

**Inspect:** maximum per-frame contrast with an explicit warning that intensity is not comparable across time

In [ ]:
case_heading("NN-GRAPH-FRAME-NORMALIZED-UPDATES", "maximum per-frame contrast with an explicit warning that intensity is not comparable across time")
display(visualize_nn_graph(xor_model,xor_X[3],xor_history,max_frames=6,frame_duration=320,evolution_mode='updates',update_reference='previous',update_scale='frame',top_k_updates=4,theme='accessible',size='wide'))

## Objective geometry and backpropagation

The surface below is an exact evaluation of the selected loss on one affine
two-direction slice through parameter space. It is not the full
high-dimensional landscape. The recorded optimization path is projected onto
that plane. The backpropagation figure separately shows the canonical chain
rule and scales the backward paths by genuine recorded layer-gradient norms.

### `NN-LOSS-LANDSCAPE-XOR`

**Inspect:** an exact BCELoss slice with the recorded XOR path projected onto it

In [ ]:
case_heading("NN-LOSS-LANDSCAPE-XOR", "an exact BCELoss slice with the recorded XOR path projected onto it")
display(visualize_nn_loss_landscape(xor_model,xor_X,torch.tensor([[0.],[1.],[1.],[0.]]),torch.nn.BCELoss(),xor_history,grid_size=17,max_frames=6,theme='academic',size='wide'))

### `NN-BACKPROP-XOR`

**Inspect:** forward equations, backward chain rule, and globally comparable recorded gradient norms

In [ ]:
case_heading("NN-BACKPROP-XOR", "forward equations, backward chain rule, and globally comparable recorded gradient norms")
display(visualize_nn_backpropagation(xor_model,xor_history,input_sample=xor_X[:1],max_frames=None,frame_duration=1100,theme='classroom',math_font_scale=1.2,size='wide'))

## Synthetic nonlinear regression

### `NN-SYNTH-REG-ARCHITECTURE`

**Inspect:** a small dense regression model in the established architecture style

In [ ]:
case_heading("NN-SYNTH-REG-ARCHITECTURE", "a small dense regression model in the established architecture style")
display(visualize_nn_architecture(reg_model,reg_X[:1],history=reg_history,theme='academic'))

### `NN-SYNTH-REG-BLOCKS`

**Inspect:** the same regression model as an execution graph with formulas

In [ ]:
case_heading("NN-SYNTH-REG-BLOCKS", "the same regression model as an execution graph with formulas")
display(visualize_nn_blocks(reg_model,reg_X[:1],theme='classroom',size='wide'))

### `NN-SYNTH-REG-TRAINING`

**Inspect:** recorded MSE optimization and regression metrics

In [ ]:
case_heading("NN-SYNTH-REG-TRAINING", "recorded MSE optimization and regression metrics")
display(visualize_nn_training(reg_history,max_frames=8,theme='academic',format='lesson'))

### `NN-SYNTH-REG-GRAPH`

**Inspect:** animated hidden activations, weights, and gradients for regression

In [ ]:
case_heading("NN-SYNTH-REG-GRAPH", "animated hidden activations, weights, and gradients for regression")
display(visualize_nn_graph(reg_model,reg_X[3],reg_history,max_frames=8,frame_duration=260))

### `NN-SYNTH-REG-PREDICTION`

**Inspect:** a final-model numerical regression substitution and prediction

In [ ]:
case_heading("NN-SYNTH-REG-PREDICTION", "a final-model numerical regression substitution and prediction")
display(explain_nn_prediction(reg_model,reg_X[3],history=reg_history,parameter_state='final',theme='academic'))

## Real Iris multiclass classification

### `NN-REAL-IRIS-ARCHITECTURE`

**Inspect:** legacy architecture with real four-feature multiclass data

In [ ]:
case_heading("NN-REAL-IRIS-ARCHITECTURE", "legacy architecture with real four-feature multiclass data")
display(visualize_nn_architecture(iris_model,iris_X[:1],history=iris_history,theme='classroom'))

### `NN-REAL-IRIS-BLOCKS`

**Inspect:** execution blocks and a three-logit output

In [ ]:
case_heading("NN-REAL-IRIS-BLOCKS", "execution blocks and a three-logit output")
display(visualize_nn_blocks(iris_model,iris_X[:1],theme='academic',size='wide'))

### `NN-REAL-IRIS-TRAINING`

**Inspect:** cross-entropy with inferred multiclass metrics

In [ ]:
case_heading("NN-REAL-IRIS-TRAINING", "cross-entropy with inferred multiclass metrics")
display(visualize_nn_training(iris_history,max_frames=8,theme='academic'))

### `NN-REAL-IRIS-WEIGHTS`

**Inspect:** real-data parameter evolution with bounded matrix display

In [ ]:
case_heading("NN-REAL-IRIS-WEIGHTS", "real-data parameter evolution with bounded matrix display")
display(visualize_nn_weights(iris_history,max_frames=8,max_rows=4,max_cols=5,theme='academic'))

### `NN-REAL-IRIS-ACTIVATIONS`

**Inspect:** recorded hidden representations for multiclass learning

In [ ]:
case_heading("NN-REAL-IRIS-ACTIVATIONS", "recorded hidden representations for multiclass learning")
display(visualize_nn(iris_model,iris_X[:1],history=iris_history,view='activations',max_frames=8,theme='accessible'))

### `NN-REAL-IRIS-GRAPH`

**Inspect:** animated neural graph for one real Iris observation

In [ ]:
case_heading("NN-REAL-IRIS-GRAPH", "animated neural graph for one real Iris observation")
display(visualize_nn_graph(iris_model,iris_X[12],iris_history,max_frames=8,theme='classroom'))

### `NN-REAL-IRIS-PREDICTION`

**Inspect:** layer-by-layer logits for one real Iris observation

In [ ]:
case_heading("NN-REAL-IRIS-PREDICTION", "layer-by-layer logits for one real Iris observation")
display(explain_nn_prediction(iris_model,iris_X[12],history=iris_history,max_frames=8,theme='academic'))

## Real breast-cancer binary classification

### `NN-REAL-CANCER-BLOCKS`

**Inspect:** a wider 30-feature binary network with dropout and sigmoid

In [ ]:
case_heading("NN-REAL-CANCER-BLOCKS", "a wider 30-feature binary network with dropout and sigmoid")
display(visualize_nn_blocks(cancer_model,cancer_X[:1],theme='accessible',size='wide'))

### `NN-REAL-CANCER-TRAINING`

**Inspect:** binary cross-entropy and inferred classification metrics

In [ ]:
case_heading("NN-REAL-CANCER-TRAINING", "binary cross-entropy and inferred classification metrics")
display(visualize_nn_training(cancer_history,max_frames=8,theme='academic'))

### `NN-REAL-CANCER-PREDICTION`

**Inspect:** a high-dimensional real-data binary forward explanation

In [ ]:
case_heading("NN-REAL-CANCER-PREDICTION", "a high-dimensional real-data binary forward explanation")
display(explain_nn_prediction(cancer_model,cancer_X[20],history=cancer_history,max_frames=7,max_neurons_math=6,theme='academic',size='wide'))

## Real handwritten digits with a convolutional network

### `NN-REAL-DIGITS-ARCHITECTURE`

**Inspect:** legacy convolution, pooling, flatten, and ten-class architecture

In [ ]:
case_heading("NN-REAL-DIGITS-ARCHITECTURE", "legacy convolution, pooling, flatten, and ten-class architecture")
display(visualize_nn_architecture(digits_model,digits_X[:1],history=digits_history,max_layers=8,theme='classroom',size='wide'))

### `NN-REAL-DIGITS-BLOCKS`

**Inspect:** shape-aware convolutional execution blocks for real 8x8 images

In [ ]:
case_heading("NN-REAL-DIGITS-BLOCKS", "shape-aware convolutional execution blocks for real 8x8 images")
display(visualize_nn_blocks(digits_model,digits_X[:1],theme='academic',size='wide'))

### `NN-REAL-DIGITS-DENSE-HEAD-GRAPH`

**Inspect:** complete executed CNN topology; convolution, normalization, pooling, reshape, and classifier are all visible

In [ ]:
case_heading("NN-REAL-DIGITS-DENSE-HEAD-GRAPH", "complete executed CNN topology; convolution, normalization, pooling, reshape, and classifier are all visible")
display(visualize_nn_graph(digits_model,digits_X[0],digits_history,max_neurons=8,max_frames=6,theme='academic',size='wide'))

### `NN-REAL-DIGITS-TRAINING`

**Inspect:** recorded ten-class CNN training metrics

In [ ]:
case_heading("NN-REAL-DIGITS-TRAINING", "recorded ten-class CNN training metrics")
display(visualize_nn_training(digits_history,max_frames=None,theme='academic'))

### `NN-REAL-DIGITS-WEIGHTS`

**Inspect:** convolution kernels and dense matrices with explicit truncation

In [ ]:
case_heading("NN-REAL-DIGITS-WEIGHTS", "convolution kernels and dense matrices with explicit truncation")
display(visualize_nn_weights(digits_history,max_frames=None,max_rows=3,max_cols=4,max_parameters=5,theme='academic',size='wide'))

## Structural and capture stress cases

### `NN-BLOCK-RESIDUAL`

**Inspect:** a residual branch and functional Add merge remain explicit

In [ ]:
case_heading("NN-BLOCK-RESIDUAL", "a residual branch and functional Add merge remain explicit")
m=ResidualNet()
x=torch.randn(2,4)
display(visualize_nn_blocks(m,x,theme='academic',size='wide'))

### `NN-BLOCK-SHARED`

**Inspect:** one shared Linear module appears as two ordered calls

In [ ]:
case_heading("NN-BLOCK-SHARED", "one shared Linear module appears as two ordered calls")
m=SharedNet()
x=torch.randn(2,4)
display(visualize_nn_blocks(m,x,show_formulas=True,size='wide'))

### `NN-BLOCK-CONV`

**Inspect:** convolution, BatchNorm buffers, activation, pooling, flatten, and classifier shapes

In [ ]:
case_heading("NN-BLOCK-CONV", "convolution, BatchNorm buffers, activation, pooling, flatten, and classifier shapes")
m=ConvNet()
x=torch.randn(2,3,8,8)
display(visualize_nn_blocks(m,x,theme='classroom',size='wide'))

### `NN-BLOCK-EMBEDDING`

**Inspect:** integer token dtype, embedding, sequence reduction, and classifier

In [ ]:
case_heading("NN-BLOCK-EMBEDDING", "integer token dtype, embedding, sequence reduction, and classifier")
m=EmbeddingNet()
tokens=torch.tensor([[1,2,3,4],[3,5,7,9]],dtype=torch.long)
display(visualize_nn_blocks(m,tokens,theme='accessible',size='wide'))

### `NN-BLOCK-LSTM`

**Inspect:** a recurrent primitive with hidden/cell outputs and public hyperparameters

In [ ]:
case_heading("NN-BLOCK-LSTM", "a recurrent primitive with hidden/cell outputs and public hyperparameters")
m=torch.nn.LSTM(5,7,num_layers=2,batch_first=True,bidirectional=True,dropout=0.1)
x=torch.randn(2,4,5)
display(visualize_nn_blocks(m,x,theme='academic',size='wide'))

### `NN-BLOCK-ATTENTION`

**Inspect:** query, key, value inputs and both attention outputs as one semantic primitive

In [ ]:
case_heading("NN-BLOCK-ATTENTION", "query, key, value inputs and both attention outputs as one semantic primitive")
m=torch.nn.MultiheadAttention(8,2,batch_first=True,dropout=0.1)
q=torch.randn(2,4,8)
display(visualize_nn_blocks(m,(q,q.clone(),q.clone()),theme='academic',size='wide'))

### `NN-BLOCK-MULTI-IO`

**Inspect:** Siamese inputs, shared encoder calls, concatenation, score, and auxiliary outputs

In [ ]:
case_heading("NN-BLOCK-MULTI-IO", "Siamese inputs, shared encoder calls, concatenation, score, and auxiliary outputs")
m=SiameseNet()
left,right=torch.randn(2,4),torch.randn(2,4)
display(visualize_nn_blocks(m,(left,right),theme='classroom',size='wide'))

### `NN-BLOCK-DYNAMIC`

**Inspect:** data-dependent Python control flow uses the disclosed eager fallback

In [ ]:
case_heading("NN-BLOCK-DYNAMIC", "data-dependent Python control flow uses the disclosed eager fallback")
m=DynamicBranch()
x=torch.ones(2,4)
display(visualize_nn_blocks(m,x,theme='accessible',size='wide'))

### `NN-BLOCK-COLLAPSE`

**Inspect:** a long network collapses visual middle nodes without changing capture

In [ ]:
case_heading("NN-BLOCK-COLLAPSE", "a long network collapses visual middle nodes without changing capture")
layers=[]
for _ in range(18):
    layers.extend([torch.nn.Linear(8,8),torch.nn.ReLU()])
m=torch.nn.Sequential(*layers)
display(visualize_nn_blocks(m,torch.randn(2,8),max_nodes=12,show_formulas=False,size='wide'))

### `NN-BLOCK-CUSTOM`

**Inspect:** a project-defined semantic descriptor extends the block vocabulary

In [ ]:
case_heading("NN-BLOCK-CUSTOM", "a project-defined semantic descriptor extends the block vocabulary")
register_neural_descriptor('Identity',role='operation',label='Pedagogical identity',formula=r'\mathbf{y}=\mathbf{x}',replace=True)
m=torch.nn.Identity()
display(visualize_nn_blocks(m,torch.randn(2,4),theme='academic'))

### `NN-RECORDER-V2`

**Inspect:** buffers, optimizer groups, adaptive state norms, and temporal phases

In [ ]:
case_heading("NN-RECORDER-V2", "buffers, optimizer groups, adaptive state norms, and temporal phases")
m=torch.nn.Sequential(torch.nn.Linear(4,4),torch.nn.BatchNorm1d(4),torch.nn.ReLU(),torch.nn.Linear(4,1))
x=torch.randn(12,4); y=torch.randn(12,1)
opt=torch.optim.Adam([{'params':m[0].parameters(),'lr':0.01},{'params':list(m[1:].parameters()),'lr':0.003}],weight_decay=0.001)
loss_fn=torch.nn.MSELoss()
r=TorchTrainingRecorder(m,optimizer=opt,loss_fn=loss_fn,capture_optimizer_state=True)
for step in range(4):
    opt.zero_grad(); prediction=m(x); loss=loss_fn(prediction,y); loss.backward(); opt.step(); r.record(step+1,loss=loss,predictions=prediction,targets=y,task='regression',capture_phase='post_step')
r.close(); h=r.to_history()
display(visualize_nn_training(h,max_frames=None,theme='academic'))

## Hundreds of neurons and many layers

Large networks use semantic blocks rather than one glyph per neuron and one
line per connection. `max_nodes` bounds the semantic block view. A graph
request automatically prefers the complete executed topology whenever a dense
neuron replay would omit Dropout, convolution, normalization, pooling, tensor
operations, or branches. Pure dense graphs still sample up to `max_neurons`
values per layer and derive marker diameter from actual pixel spacing. This
keeps the visualization inspectable without presenting an incomplete network.

### `NN-LARGE-HUNDREDS-MANY-LAYERS`

**Inspect:** a trained 128-to-512 network fixture reused by every supported neural figure

In [ ]:
case_heading("NN-LARGE-HUNDREDS-MANY-LAYERS", "a trained 128-to-512 network fixture reused by every supported neural figure")
widths=[128,512,384,256,128,64,32,10]
layers=[]
for left,right in zip(widths[:-1],widths[1:]):
    layers.extend([torch.nn.Linear(left,right),torch.nn.GELU(),torch.nn.Dropout(0.1)])
large_model=torch.nn.Sequential(*layers[:-1])
large_input=torch.randn(16,128)
large_target=torch.randint(0,10,(16,))
large_loss_fn=torch.nn.CrossEntropyLoss()
large_optimizer=torch.optim.Adam(large_model.parameters(),lr=0.002)
large_recorder=TorchTrainingRecorder(large_model,optimizer=large_optimizer,loss_fn=large_loss_fn,max_tensor_elements=300000,max_activation_elements=1024)
for step in range(6):
    large_optimizer.zero_grad(); large_prediction=large_model(large_input); large_loss=large_loss_fn(large_prediction,large_target); large_loss.backward(); large_optimizer.step()
    large_recorder.record(step+1,loss=large_loss,predictions=large_prediction,targets=large_target,task='classification')
large_recorder.close(); large_history=large_recorder.to_history()
display(visualize_nn_architecture(large_model,large_input[:1],history=large_history,max_layers=8,theme='academic',size='wide'))

### `NN-LARGE-ARCHITECTURE`

**Inspect:** bounded mathematical architecture for hundreds of units

In [ ]:
case_heading("NN-LARGE-ARCHITECTURE", "bounded mathematical architecture for hundreds of units")
display(visualize_nn_architecture(large_model,large_input[:1],history=large_history,max_layers=8,theme='academic',size='wide'))

### `NN-LARGE-BLOCKS`

**Inspect:** execution graph with a concise collapsed summary node

In [ ]:
case_heading("NN-LARGE-BLOCKS", "execution graph with a concise collapsed summary node")
display(visualize_nn_blocks(large_model,large_input[:1],max_nodes=18,show_formulas=False,theme='academic',size='wide'))

### `NN-LARGE-GRAPH`

**Inspect:** complete executed topology for the full deep network, including every Dropout stage

In [ ]:
case_heading("NN-LARGE-GRAPH", "complete executed topology for the full deep network, including every Dropout stage")
display(visualize_nn_graph(large_model,large_input[0],large_history,max_neurons=8,max_frames=6,show_loss_panel=True,theme='academic',size='wide'))

### `NN-LARGE-FORWARD-SUBSTITUTION`

**Inspect:** input, bounded layer substitutions, and ten-logit output

In [ ]:
case_heading("NN-LARGE-FORWARD-SUBSTITUTION", "input, bounded layer substitutions, and ten-logit output")
display(explain_nn_prediction(large_model,large_input[0],history=large_history,max_layers_math=6,max_neurons_math=6,max_frames=6,theme='academic',size='wide'))

### `NN-LARGE-TRAINING`

**Inspect:** cross-entropy and multiclass metrics

In [ ]:
case_heading("NN-LARGE-TRAINING", "cross-entropy and multiclass metrics")
display(visualize_nn_training(large_history,max_frames=None,theme='academic',size='wide'))

### `NN-LARGE-WEIGHTS`

**Inspect:** bounded parameter matrices without reducing mathematical type

In [ ]:
case_heading("NN-LARGE-WEIGHTS", "bounded parameter matrices without reducing mathematical type")
display(visualize_nn_weights(large_history,max_rows=3,max_cols=4,max_parameters=6,max_frames=6,theme='academic',size='wide'))

### `NN-LARGE-ACTIVATIONS`

**Inspect:** recorded layer activation summaries

In [ ]:
case_heading("NN-LARGE-ACTIVATIONS", "recorded layer activation summaries")
display(visualize_nn(large_model,large_input[:1],history=large_history,view='activations',max_frames=6,theme='accessible',size='wide'))

### `NN-LARGE-BACKPROPAGATION`

**Inspect:** bounded per-layer gradients, updates, relative changes, and loss effect

In [ ]:
case_heading("NN-LARGE-BACKPROPAGATION", "bounded per-layer gradients, updates, relative changes, and loss effect")
display(visualize_nn_backpropagation(large_model,large_history,input_sample=large_input[:1],max_layers=6,max_frames=6,frame_duration=1100,theme='classroom',size='wide'))

### `NN-LARGE-LOSS-LANDSCAPE`

**Inspect:** exact reduced-grid two-direction CrossEntropyLoss slice for the large model

In [ ]:
case_heading("NN-LARGE-LOSS-LANDSCAPE", "exact reduced-grid two-direction CrossEntropyLoss slice for the large model")
display(visualize_nn_loss_landscape(large_model,large_input,large_target,large_loss_fn,large_history,grid_size=9,max_frames=4,theme='academic',size='wide'))